# CPMM TPU Code LLM

This notebook trains a pure post-transformer CPMM-style Python code model on Colab TPU with Google Drive checkpointing and exact resume support.

Notebook flow:
- install TPU/JAX dependencies
- mount Google Drive
- prepare tokenizer and JSONL training shards
- initialize or restore the CPMM JAX model
- pretrain on Python code with next-token and graph-memory objectives
- optionally instruction-tune for GPT-style chat/code behavior
- run long-context code evaluations and sample generations

In [ ]:
# Colab TPU setup
# Run this cell first after selecting a TPU runtime.

!pip install -q -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q flax optax orbax-checkpoint datasets sentencepiece etils

import os
import sys
from pathlib import Path

import jax
print(jax.devices())

REPO_ROOT = Path('/content/zap')
if not REPO_ROOT.exists():
    raise FileNotFoundError('Clone or copy the repository into /content/zap before running the notebook.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# Google Drive mount and experiment config

from google.colab import drive

drive.mount('/content/drive')

from cpmm_jax.config import ExperimentConfig, save_config

config = ExperimentConfig()
config.checkpoint.drive_root = '/content/drive/MyDrive/cpmm_code_llm'
config.checkpoint.checkpoint_dir = '/content/drive/MyDrive/cpmm_code_llm/checkpoints'
config.checkpoint.metadata_path = '/content/drive/MyDrive/cpmm_code_llm/checkpoints/metadata.json'
config.data.cache_dir = '/content/drive/MyDrive/cpmm_code_llm/cache'
config.data.tokenizer_path = '/content/drive/MyDrive/cpmm_code_llm/tokenizer.model'
config.data.train_shards_glob = '/content/drive/MyDrive/cpmm_code_llm/data/train/*.jsonl'
config.data.eval_shards_glob = '/content/drive/MyDrive/cpmm_code_llm/data/eval/*.jsonl'

Path(config.checkpoint.drive_root).mkdir(parents=True, exist_ok=True)
save_config(config, Path(config.checkpoint.drive_root) / 'experiment_config.json')
config

In [ ]:
# Tokenizer + permissive Python shard preparation
# This cell is resumable: if shard files already exist, it skips work.

import glob
import json
from pathlib import Path

from datasets import load_dataset
import sentencepiece as spm

from cpmm_jax.data_pipeline import build_training_record, write_jsonl_shards

TOKENIZER_PREFIX = Path(config.data.tokenizer_path).with_suffix('')
TRAIN_DIR = Path('/content/drive/MyDrive/cpmm_code_llm/data/train')
EVAL_DIR = Path('/content/drive/MyDrive/cpmm_code_llm/data/eval')
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

if not Path(config.data.tokenizer_path).exists():
    # Replace this dataset call with your approved permissive Python subset manifest if needed.
    ds = load_dataset(config.data.dataset_name, split='train', streaming=True)
    text_dump = Path('/content/drive/MyDrive/cpmm_code_llm/tokenizer_corpus.txt')
    with text_dump.open('w', encoding='utf-8') as handle:
        for i, row in enumerate(ds):
            lang = row.get('lang') or row.get('language') or row.get('programming_language')
            if lang and str(lang).lower() != 'python':
                continue
            content = row.get('content') or row.get('code') or ''
            if not content:
                continue
            handle.write(content.replace('\x00', ' ') + '\n')
            if i >= 20000:
                break
    spm.SentencePieceTrainer.train(
        input=str(text_dump),
        model_prefix=str(TOKENIZER_PREFIX),
        vocab_size=config.model.vocab_size,
        model_type='unigram',
        character_coverage=1.0,
        bos_id=config.data.bos_id,
        eos_id=config.data.eos_id,
        pad_id=config.data.pad_id,
        unk_id=config.data.unk_id,
    )

sp = spm.SentencePieceProcessor(model_file=config.data.tokenizer_path)

class SentencePieceAdapter:
    def __init__(self, processor):
        self.processor = processor
    def encode(self, text: str):
        return self.processor.encode(text, out_type=int)

tokenizer = SentencePieceAdapter(sp)

if not glob.glob(str(TRAIN_DIR / '*.jsonl')):
    ds = load_dataset(config.data.dataset_name, split='train', streaming=True)
    train_records = []
    eval_records = []
    for i, row in enumerate(ds):
        lang = row.get('lang') or row.get('language') or row.get('programming_language')
        if lang and str(lang).lower() != 'python':
            continue
        content = row.get('content') or row.get('code') or ''
        if not content or len(content) < 64:
            continue
        record = build_training_record(
            content,
            tokenizer=tokenizer,
            config=config.data,
            metadata={'repo': row.get('repo_name', ''), 'path': row.get('path', '')},
        )
        if len(eval_records) < 512:
            eval_records.append(record)
        else:
            train_records.append(record)
        if len(train_records) >= 20000:
            break
    write_jsonl_shards(train_records, TRAIN_DIR, examples_per_shard=512, prefix='train')
    write_jsonl_shards(eval_records, EVAL_DIR, examples_per_shard=256, prefix='eval')

print('train shards', len(glob.glob(str(TRAIN_DIR / '*.jsonl'))))
print('eval shards', len(glob.glob(str(EVAL_DIR / '*.jsonl'))))

In [ ]:
# Initialize or resume CPMM training state

import itertools
import numpy as np
import orbax.checkpoint as ocp

from cpmm_jax.checkpointing import (
    create_checkpoint_manager,
    latest_step,
    load_lightweight_metadata,
    metadata_payload,
    restore_checkpoint,
    save_checkpoint,
    save_lightweight_metadata,
)
from cpmm_jax.data_pipeline import JsonlShardLoader, LoaderState, collate_batch, load_loader_state
from cpmm_jax.train_step import create_train_state

rng = jax.random.PRNGKey(config.train.seed)
state, model = create_train_state(rng, config.model, config.cpmm, config.train)
manager = create_checkpoint_manager(config.checkpoint, config.train.max_checkpoints_to_keep)
loader_state = LoaderState(rng_seed=config.train.seed)
global_step = 0
start_epoch = 0

latest = latest_step(manager)
meta = load_lightweight_metadata(config.checkpoint)
if latest is not None and meta is not None:
    state, restored_meta = restore_checkpoint(manager, latest, state)
    global_step = int(restored_meta['step'])
    start_epoch = int(restored_meta['epoch'])
    loader_state = LoaderState(**restored_meta['loader_state'])
    print(f'Resumed from checkpoint step {global_step}')
else:
    print('Starting fresh training run')

train_loader = JsonlShardLoader(config.data.train_shards_glob, seed=config.train.seed)
eval_loader = JsonlShardLoader(config.data.eval_shards_glob, seed=config.train.seed + 1)

In [ ]:
# CPMM pretraining loop with exact resume support

import jax.numpy as jnp
from cpmm_jax.train_step import eval_step, train_step


def batch_iterator(loader, state_obj, batch_size):
    pending = []
    current_state = state_obj
    for next_state, record in loader.iter_records(current_state):
        pending.append(record)
        current_state = next_state
        if len(pending) == batch_size:
            yield current_state, collate_batch(pending, config.model.max_seq_len, config.data.pad_id)
            pending = []


def to_device(batch):
    return {key: jnp.asarray(value) for key, value in batch.items()}

train_iter = batch_iterator(train_loader, loader_state, config.train.global_batch_size)
for step in range(global_step, config.train.total_steps):
    loader_state, batch = next(train_iter)
    batch = to_device(batch)
    state, metrics = train_step(state, model, batch, config.cpmm)

    if step % config.train.log_every == 0:
        printable = {k: float(v) for k, v in metrics.items()}
        print(f'step={step}', printable)

    if step % config.train.eval_every == 0 and step > global_step:
        eval_state, eval_batch = next(batch_iterator(eval_loader, LoaderState(rng_seed=config.train.seed + 1), config.train.global_batch_size))
        del eval_state
        eval_metrics = eval_step(state, model, to_device(eval_batch), config.cpmm)
        print('eval', {k: float(v) for k, v in eval_metrics.items()})

    if step % config.train.save_every == 0 and step > global_step:
        payload = metadata_payload(
            step=step,
            epoch=start_epoch,
            rng_seed=config.train.seed,
            loader_state=loader_state,
            tokenizer_path=config.data.tokenizer_path,
            stage='pretrain',
        )
        save_checkpoint(manager, step, state, payload)
        save_lightweight_metadata(config.checkpoint, payload)

print('pretraining complete')

In [ ]:
# Optional chat/code supervised fine-tuning

from cpmm_jax.finetune_chat import ChatExample, build_chat_corpus

chat_examples = [
    ChatExample(
        system=config.chat.system_prompt,
        user='Write a Python function that returns the Fibonacci sequence up to n.',
        assistant='def fib_upto(n):\n    seq = []\n    a, b = 0, 1\n    while a <= n:\n        seq.append(a)\n        a, b = b, a + b\n    return seq',
    ),
    ChatExample(
        system=config.chat.system_prompt,
        user='Explain what this code does: def add(a, b): return a + b',
        assistant='It defines a function named add that takes two arguments and returns their sum.',
    ),
]
chat_corpus = build_chat_corpus(chat_examples)
print(chat_corpus[0][:300])
print('Add your full instruction dataset and a chat fine-tuning loop here using the same checkpoint manager.')

In [ ]:
# Long-context code evaluation hooks

from cpmm_jax.eval_code_tasks import CodeEvalSample, aggregate_scores

samples = [
    CodeEvalSample(
        prompt='Find the definition used by helper() in a long file.',
        expected='def helper(x):\n    return x + 1',
        task_type='definition_lookup',
    ),
    CodeEvalSample(
        prompt='Trace the imported symbol used by process().',
        expected='from utils import normalize',
        task_type='import_reasoning',
    ),
]
predictions = [sample.expected for sample in samples]
aggregate_scores(samples, predictions)

## Resume notes

If Colab disconnects:
1. Reconnect to TPU runtime.
2. Run setup and Drive mount cells again.
3. Run the resume cell.
4. Run the pretraining loop cell again.

The notebook restores:
- model parameters
- optimizer state
- tokenizer path
- global step
- shard index
- sample offset
- training stage metadata